# Experiencing a Free-of-Charge Way to Train Deep Models with GPU (PyTorch Edition)
- **Tác giả gốc**: Nguyen Hai Duong (2019/02/03)
- **Cập nhật sang PyTorch**: Thành viên 1 — Lý Hồng Hưng (MSSV: 3123411132)
- **Mục đích**: Hướng dẫn sử dụng Google Colab với GPU, cài đặt thư viện PyTorch, kết nối Google Drive và đọc dữ liệu/lưu checkpoint.

In [1]:
# Thông số phần cứng tiêu biểu trên Google Colab (phiên bản miễn phí hiện tại):
# - GPU: 1x NVIDIA Tesla T4 (15 GB GDDR6 VRAM, Compute Capability 7.5)
# - CPU: Intel Xeon (2 vCPU)
# - RAM: ~12.7 GB
# - Disk: ~78 - 100 GB (tạm thời, bị reset sau tối đa ~12 giờ hoặc khi ngắt kết nối)

In [2]:
# Chọn Python 3 và GPU trên Google Colab:
# Menu [Runtime] -> [Change runtime type] -> Hardware accelerator: [T4 GPU] -> [Save]

In [3]:
# Google Colab cho phép chạy lệnh Linux bằng dấu ! ở đầu dòng
# Kiểm tra hệ điều hành Linux trên máy ảo Colab
!uname -a

Linux ca439ecc9dce 6.6.159+ #1 SMP Thu Oct 22 03:07:54 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux


In [4]:
# Kiểm tra thông tin GPU NVIDIA bằng lệnh nvidia-smi
!nvidia-smi

Thu Nov 26 02:02:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# Cài đặt các gói cần thiết (Google Colab thường đã cài sẵn torch và torchvision)
# Cài thêm torchinfo để xem tóm tắt kiến trúc mô hình tương tự model.summary() trong Keras
!pip install -q torch torchvision torchinfo

In [6]:
# Kiểm tra phiên bản PyTorch và trạng thái GPU CUDA
import torch
import torchvision

print(f"PyTorch version    : {torch.__version__}")
print(f"TorchVision version: {torchvision.__version__}")
print(f"CUDA available     : {torch.cuda.is_available()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device       : {device}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU [{i}]: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB | Compute: {props.major}.{props.minor}")

PyTorch version    : 2.14.1+cu130
TorchVision version: 0.29.1+cu130
CUDA available     : True
Using device       : cuda
  GPU [0]: Tesla T4 | VRAM: 14.56 GB | Compute: 7.5


In [7]:
# Kết nối Google Drive để lưu trữ dữ liệu và checkpoint lâu dài
# (Tránh mất dữ liệu khi phiên làm việc Colab bị reset sau ~12 giờ)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# Sau khi mount, Google Drive nằm tại đường dẫn "/content/drive/MyDrive"
# Kiểm tra nhanh thư mục gốc của Google Drive
!ls -la "/content/drive/MyDrive" | head -n 20

total 412565
drwx------ 4 root root      4096 Nov 26 02:03 .
drwx------ 5 root root      4096 Nov 26 02:03 ..
-rw------- 1 root root       179 Oct 22 10:38 3123411132_Lý Hồng Hưng (1).gdoc
-rw------- 1 root root       179 Nov 23 09:38 3123411132_Lý Hồng Hưng.gdoc
-rw------- 1 root root       179 Oct  8 10:55 3123411132_Lý Hông Hưng.gdoc
-rw------- 1 root root     53081 Sep 17  2025 4.19.png
-rw------- 1 root root  13967359 May 17  2026 BC_3123411101.docx
-rw------- 1 root root       179 Jan 21  2026 C2 - Overview of Distributed Database Systems.gdoc
-rw------- 1 root root       179 Oct  6  2024 Chapter 1_Operating system overview.gdoc
-rw------- 1 root root       179 Feb 11  2025 Chapter 2. Pointers.gdoc
-rw------- 1 root root       179 Sep 18  2024 Chapter 2_Process management (1).gdoc
-rw------- 1 root root       179 Oct  6  2024 Chapter 2_Process management.gdoc
-rw------- 1 root root    998662 Nov  3  2024 CHUONG4.zip
drwx------ 2 root root      4096 Aug 30  2021 Classro

In [9]:
# Ví dụ đọc một ảnh từ Google Drive và hiển thị bằng OpenCV + Matplotlib
# Lưu ý: Hãy thay đổi đường dẫn image_path tới một file ảnh có thật trên Drive của bạn
import os
import matplotlib.pyplot as plt
import cv2

image_path = '/content/drive/MyDrive/sample.jpg'  # Thay bằng đường dẫn ảnh thật trên Drive

if os.path.exists(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(img)
    plt.axis('off')
    plt.title('Image loaded from Google Drive')
    plt.show()
else:
    print(f"Không tìm thấy ảnh tại '{image_path}'. Hãy cập nhật đường dẫn tới ảnh trên Google Drive của bạn.")

Không tìm thấy ảnh tại '/content/drive/MyDrive/sample.jpg'. Hãy cập nhật đường dẫn tới ảnh trên Google Drive của bạn.


In [10]:
# Lưu ý: Trong PyTorch, bạn có thể lưu trọng số mô hình (checkpoint) trực tiếp lên Google Drive:
# torch.save(model.state_dict(), '/content/drive/MyDrive/model_weights.pth')
# Và nạp lại khi cần:
# model.load_state_dict(torch.load('/content/drive/MyDrive/model_weights.pth', map_location=device))